# Stage 3 CLIP Hybrid Sweep Clean (Kaggle)

Exploratory coarse-only sweep over CLIP embeddings, simple classifiers, and flashover thresholds.


In [ ]:
import json
import shutil
import subprocess
import traceback
from pathlib import Path
from collections import Counter

RUN_NAME = "stage3_clip_hybrid_sweep_clean"
WORK_DIR = Path("/kaggle/working") / RUN_NAME
ARCHIVE_PATH = Path("/kaggle/working/stage3_deliverables_clip_hybrid_sweep_clean.tar.gz")
LOG_PATH = WORK_DIR / "run_log.txt"
ERROR_PATH = WORK_DIR / "error_traceback.txt"

DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/kostyaryazanov/idid-coco-v3"),
    Path("/kaggle/input/idid-coco-v3"),
]
STAGE3_REL = Path("stage3_regrouped_v2")
TRAIN_JSONL_REL = STAGE3_REL / "train_balanced/vlm_labels_v1_train_balanced_v2.annotated.jsonl"
VAL_JSONL_REL = STAGE3_REL / "val/vlm_labels_v1_val_v2.annotated.jsonl"

CLIP_MODEL_ID = "openai/clip-vit-base-patch32"
LABELS = ["insulator_ok", "defect_flashover", "defect_broken"]
TEXT_PROMPTS = {
    "insulator_ok": "a clear photo crop of an intact power line insulator with regular shape and no visible damage",
    "defect_flashover": "a photo crop of a power line insulator with flashover burn marks dark surface traces or localized surface damage",
    "defect_broken": "a photo crop of a broken power line insulator with missing fragment damaged edge or structural break",
}

def log(msg):
    print(msg, flush=True)
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(str(msg) + "\n")

def sh(args, cwd: Path | None = None, check: bool = True):
    log("$ " + " ".join(str(a) for a in args))
    p = subprocess.run([str(a) for a in args], cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if p.stdout:
        log(p.stdout)
    if p.stderr:
        log(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(str(a) for a in args)}")
    return p

def load_jsonl(path: Path):
    rows=[]
    with path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def resolve_crop(row, split_root: Path) -> Path:
    p = Path(row["crop_path"])
    candidates = []
    if p.is_absolute():
        candidates.append(p)
    candidates.extend([
        split_root / p,
        split_root.parent / p,
        split_root.parent / "crops" / row.get("split", "") / row.get("coarse_class", "") / p.name,
    ])
    for cand in candidates:
        if cand.exists():
            return cand
    raise FileNotFoundError(f"crop not found for {row.get('record_id')}: {row.get('crop_path')}")

def package_outputs():
    try:
        if ARCHIVE_PATH.exists():
            ARCHIVE_PATH.unlink()
        sh(["tar", "-czf", str(ARCHIVE_PATH), "-C", str(WORK_DIR.parent), RUN_NAME], check=False)
        log(f"Archive: {ARCHIVE_PATH}")
    except Exception:
        print(traceback.format_exc(), flush=True)

try:
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    log(f"RUN_NAME: {RUN_NAME}")
    sh(["nvidia-smi"], check=False)

    DATA_ROOT = None
    for root in DATASET_ROOT_CANDIDATES:
        if (root / TRAIN_JSONL_REL).exists() and (root / VAL_JSONL_REL).exists():
            DATA_ROOT = root
            break
    if DATA_ROOT is None:
        raise FileNotFoundError("Could not find stage3_regrouped_v2 train/val JSONL in Kaggle inputs")
    train_jsonl = DATA_ROOT / TRAIN_JSONL_REL
    val_jsonl = DATA_ROOT / VAL_JSONL_REL
    train_root = train_jsonl.parent
    val_root = val_jsonl.parent
    log(f"DATA_ROOT: {DATA_ROOT}")

    sh(["python", "-m", "pip", "install", "-q", "transformers==4.57.1", "scikit-learn", "tabulate"])

    import numpy as np
    import pandas as pd
    import torch
    from PIL import Image
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
    from sklearn.preprocessing import normalize
    from transformers import CLIPModel, CLIPProcessor

    train_rows = [r for r in load_jsonl(train_jsonl) if r.get("coarse_class") in LABELS]
    val_rows = [r for r in load_jsonl(val_jsonl) if r.get("coarse_class") in LABELS]
    log(f"train rows: {len(train_rows)} {Counter(r['coarse_class'] for r in train_rows)}")
    log(f"val rows: {len(val_rows)} {Counter(r['coarse_class'] for r in val_rows)}")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    log(f"device: {device}")
    model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(device)
    processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
    model.eval()

    def embed_images(rows, split_root: Path, batch_size: int = 32):
        feats=[]; labels=[]; ids=[]
        with torch.no_grad():
            for start in range(0, len(rows), batch_size):
                batch = rows[start:start+batch_size]
                images=[Image.open(resolve_crop(row, split_root)).convert("RGB") for row in batch]
                inputs = processor(images=images, return_tensors="pt", padding=True).to(device)
                feats.append(model.get_image_features(**inputs).detach().cpu().numpy())
                labels.extend([r["coarse_class"] for r in batch])
                ids.extend([r["record_id"] for r in batch])
        return normalize(np.vstack(feats)), np.array(labels), ids

    def embed_texts():
        with torch.no_grad():
            inputs = processor(text=[TEXT_PROMPTS[x] for x in LABELS], return_tensors="pt", padding=True).to(device)
            return normalize(model.get_text_features(**inputs).detach().cpu().numpy())

    X_train, y_train, train_ids = embed_images(train_rows, train_root)
    X_val, y_val, val_ids = embed_images(val_rows, val_root)
    T = embed_texts()
    zs_sim = X_val @ T.T
    zs_pred = np.array([LABELS[i] for i in zs_sim.argmax(axis=1)])

    methods = []
    pred_table = {"record_id": val_ids, "gt": y_val.tolist(), "clip_zero_shot": zs_pred.tolist()}

    def add_method(name, pred, extra=None):
        pred = np.array(pred)
        row = {
            "method": name,
            "accuracy": accuracy_score(y_val, pred),
            "macro_f1_3class": f1_score(y_val, pred, labels=LABELS, average="macro", zero_division=0),
            "ok_recall": ((pred == "insulator_ok") & (y_val == "insulator_ok")).sum() / max((y_val == "insulator_ok").sum(), 1),
            "flashover_recall": ((pred == "defect_flashover") & (y_val == "defect_flashover")).sum() / max((y_val == "defect_flashover").sum(), 1),
            "broken_recall": ((pred == "defect_broken") & (y_val == "defect_broken")).sum() / max((y_val == "defect_broken").sum(), 1),
        }
        if extra:
            row.update(extra)
        methods.append(row)
        pred_table[name] = pred.tolist()

    add_method("clip_zero_shot", zs_pred)

    fitted = {}
    for class_weight in [None, "balanced"]:
        for C in [0.03, 0.1, 0.3, 1.0, 3.0, 10.0]:
            clf = LogisticRegression(max_iter=3000, class_weight=class_weight, C=C)
            clf.fit(X_train, y_train)
            pred = clf.predict(X_val)
            name = f"logreg_c{C:g}_{'balanced' if class_weight else 'plain'}"
            add_method(name, pred, {"C": C, "class_weight": str(class_weight)})
            fitted[name] = clf
            proba = clf.predict_proba(X_val)
            cls = list(clf.classes_)
            if "defect_flashover" in cls:
                flash_idx = cls.index("defect_flashover")
                base = pred.copy()
                for thr in [0.35, 0.45, 0.55, 0.65, 0.75]:
                    gated = np.array(["defect_flashover" if p[flash_idx] >= thr else "insulator_ok" for p in proba])
                    add_method(f"{name}_flash_gate_{thr:.2f}_else_ok", gated, {"C": C, "class_weight": str(class_weight), "flash_thr": thr})

    # Tiny ensemble with zero-shot broken rescue: only test whether zero-shot contains useful broken signal.
    for base_name, clf in list(fitted.items()):
        pred = np.array(pred_table[base_name])
        rescued = pred.copy()
        rescued[zs_pred == "defect_broken"] = "defect_broken"
        add_method(base_name + "_zs_broken_rescue", rescued)

    summary = pd.DataFrame(methods).sort_values(["macro_f1_3class", "accuracy"], ascending=False)
    summary.to_csv(WORK_DIR / "clip_hybrid_sweep_summary.csv", index=False)
    pd.DataFrame(pred_table).to_csv(WORK_DIR / "clip_hybrid_sweep_predictions.csv", index=False)

    best = summary.iloc[0].to_dict()
    log("Best method:")
    log(best)
    best_name = best["method"]
    best_pred = np.array(pred_table[best_name])
    report = classification_report(y_val, best_pred, labels=LABELS, zero_division=0)
    cm = confusion_matrix(y_val, best_pred, labels=LABELS)
    (WORK_DIR / "best_classification_report.txt").write_text(report, encoding="utf-8")
    pd.DataFrame(cm, index=LABELS, columns=LABELS).to_csv(WORK_DIR / "best_confusion_matrix.csv")

    lines = [
        "# Stage 3 CLIP Hybrid Sweep Clean",
        "",
        f"CLIP model: `{CLIP_MODEL_ID}`",
        "",
        "This is an exploratory coarse-only sweep over simple classifiers and thresholds. It uses val labels for comparison, so it should guide hypotheses rather than define a final tuned method.",
        "",
        "## Top 12 by macro-F1",
        "",
        summary.head(12).to_markdown(index=False),
        "",
        "## Best classification report",
        "",
        "```text",
        report,
        "```",
    ]
    (WORK_DIR / "summary.md").write_text("\n".join(lines), encoding="utf-8")
    log(summary.head(12).to_string(index=False))
    package_outputs()
except Exception:
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    tb = traceback.format_exc()
    ERROR_PATH.write_text(tb, encoding="utf-8")
    print(tb, flush=True)
    package_outputs()
    raise
